In [ ]:
!pip install gradio
!pip install transformers datasets torch

## **Libraries**

In [ ]:
import re
import torch
import gradio as gr
import json
from transformers import BertTokenizer, BertForMaskedLM

## **Bert Model**

In [ ]:
# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

# **Loading and cleaning DataSet**

In [ ]:
def load_clean_text(path):
    with open(path, encoding='utf-8') as file:
        text = file.read()

    # Remove Gutenberg header & footer
    start = text.find("*** START OF")
    end = text.find("*** END OF")
    if start != -1 and end != -1:
        text = text[start:end]

    # Lowercase
    text = text.lower()

    # Remove special characters (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Split into lines (optional, similar to your previous format)
    lines = text.split('.')

    print(f"Loaded {len(lines)} lines with {len(text.split())} words.")

    return lines

# Usage
file_path = "/content/1661-0.txt"
corpus = load_clean_text(file_path)

Loaded 1 lines with 105897 words.


# **Custom Tokenizer and saving it in JASON file**

In [ ]:
# Custom tokenizer function (whitespace + lowercasing + basic cleanup)
def custom_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    tokens = text.strip().split()
    return tokens

# Save tokenized corpus to a JSON file
tokenized_corpus = [custom_tokenize(line) for line in corpus]
with open("tokenized_movie_lines.json", "w", encoding="utf-8") as f:
    json.dump(tokenized_corpus, f, ensure_ascii=False, indent=2)
print("Tokenized corpus saved to 'tokenized_movie_lines.json'")

Tokenized corpus saved to 'tokenized_movie_lines.json'


# **Word Prediction**

In [ ]:
# BERT-based prediction
def predict_next_words(seed_text, top_k=5):
    seed_text = seed_text.strip()
    if not seed_text.endswith(" "):
        seed_text += " "
    input_text = seed_text + "[MASK]"
    input_ids = tokenizer.encode(input_text, return_tensors='pt')
    mask_index = torch.where(input_ids == tokenizer.mask_token_id)[1]

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits

    mask_word_logits = logits[0, mask_index, :]
    sorted_indices = torch.argsort(mask_word_logits, dim=-1, descending=True)[0]

    predictions = []
    for idx in sorted_indices:
        word = tokenizer.decode([idx]).strip()
        if word.isalpha():
            predictions.append(word)
        if len(predictions) == top_k:
            break

    top_prediction = predictions[0] if predictions else "(no prediction)"
    return f"Top prediction: {top_prediction}\nTop {top_k} predictions: {predictions}"

# **Interface**

In [ ]:
# Gradio interface
iface = gr.Interface(
    fn=predict_next_words,
    inputs=gr.Textbox(lines=2, placeholder="Enter a 3+ word phrase..."),
    outputs="text",
    title="Text Auto complete NLP Project using BERT",
    description="Enter a phrase (3+ words) and get BERT's top next-word suggestions trained on movie dialogues."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6fb6cce1a5f232eb5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
